# Connect to RabbitMQ

In [57]:
import numpy as np
from communication import protocol
from communication.rabbitmq import Rabbitmq

# Initialize RabbitMQ connection (adjust parameters as needed)
try:
    rmq = Rabbitmq(
        ip="localhost",
        port=5672,
        username="ur3e",
        password="ur3e",
        vhost="/",
        exchange="UR3E_AMQP",
        type="topic",
    )
    rmq.connect_to_server()
    print("✓ Connected to RabbitMQ successfully")
except Exception as e:
    print(f"✗ Failed to connect to RabbitMQ: {e}")
    print("\nMake sure RabbitMQ is running. You can start it with:")
    print("  python -m startup.start_docker_rabbitmq")

def send_control_message(rmq, msg):
    """Send a control message to the UR3e Mockup via RabbitMQ."""
    try:
        rmq.send_message(
            routing_key=protocol.ROUTING_KEY_CTRL,
            message=msg
        )
        print(f"✓ Control message: {msg} sent successfully")
    except Exception as e:
        print(f"✗ Failed to send control message: {e}")

✓ Connected to RabbitMQ successfully


# Random movement generator

The block below will create random movements. Every time the arm has finished a movement press ctrl+enter to generate a new movement.

In [58]:
# Construct control message for loading a program
position_np = (np.random.rand((6)) - 0.5) * 0.5*np.pi * 1000 # Generate 6 random joint positions
position_np = np.ones((6))*0*np.pi
position = position_np.tolist()
vel = 60 # deg/s
acc = 1 # deg/s²

msg = {
    protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.LOAD_PROGRAM,
    protocol.CtrlMsgKeys.JOINT_POSITIONS: [position],
    protocol.CtrlMsgKeys.MAX_VELOCITY: vel,
    protocol.CtrlMsgKeys.ACCELERATION: acc,
}

send_control_message(rmq, msg)

# send control message for starting program
msg_start = {
    protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.PLAY,
}

send_control_message(rmq, msg_start)


✓ Control message: {'type': 'load_program', 'joint_positions': [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], 'max_velocity': 60, 'acceleration': 1} sent successfully
✓ Control message: {'type': 'play'} sent successfully


# Automatic movement generator

In [27]:
import time

while True:
    # Construct control message for loading a program
    position_np = (np.random.normal(0, 1, (6)) - 0.5) * 2*np.pi # Generate 6 random joint positions [-pi, pi]
    position = position_np.tolist()

    max_vel_bounds = (0, 360) # deg/s
    max_acc_bounds = (0, 180) # deg/s²      *Assumed

    def random_from_range(range: tuple):
        return np.random.rand() * (range[1] - range[0]) + range[0]

    vel = random_from_range(max_vel_bounds) # deg/s
    acc = random_from_range(max_acc_bounds) # deg/s²

    msg = {
        protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.LOAD_PROGRAM,
        protocol.CtrlMsgKeys.JOINT_POSITIONS: [position],
        protocol.CtrlMsgKeys.MAX_VELOCITY: vel,
        protocol.CtrlMsgKeys.ACCELERATION: acc,
    }

    send_control_message(rmq, msg)

    # send control message for starting program
    msg_start = {
        protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.PLAY,
    }

    send_control_message(rmq, msg_start)

    time.sleep(100)

# TODO:
# - Should generate random movement CTRL messages
# - The movements should be uniformly distributed
# - A new message should only be sent after the previous motion is done (Wait for stationarity)
  # - Lowpass filter over past N states, if velocity approx 0, send new control message
# - The messages should contain random joint positions, max velocity and acceleration


✓ Control message: {'type': 'load_program', 'joint_positions': [[-2.060981139108012, -0.8228253521051845, -5.4965014523021365, -5.973270841773573, -3.2025008118846587, -15.414080473941521]], 'max_velocity': 14.69763388497495, 'acceleration': 88.67453354426901} sent successfully
✓ Control message: {'type': 'play'} sent successfully


KeyboardInterrupt: 